# DAQ-BP: Degeneracy-Aware Quantized Belief Propagation
### Interactive Execution, Validation & Benchmarking Notebook

**Author:** Deeven Seru  
**Repository:** [github.com/Deeven-Seru/daq-bp](https://github.com/Deeven-Seru/daq-bp)

This notebook executes an interactive demonstration of the **Degeneracy-Aware Quantized Belief Propagation (DAQ-BP)** decoder on the canonical **IBM $[[144, 12, 12]]$ Bivariate Bicycle Gross Code** ($n=144, k=12, d=12$).

#### What this notebook verifies live:
1. **Gross Code Construction & CSS Orthogonality:** Algebraically constructs $H_X, H_Z$ and checks $H_X H_Z^T \equiv 0 \pmod 2$.
2. **5-bit Integer Quantization & Dynamic LUTs:** Visualizes the 5-bit integer lookup tables that replace continuous $\tanh / \text{atanh}$ functions.
3. **Live Single-Shot Decoding Walkthrough:** Injects a multi-qubit Pauli-X error and tracks message passing step-by-step.
4. **Comparative Monte Carlo Benchmark:** Evaluates Logical Error Rate (LER) and convergence iterations across:
   - Standalone Float-BP (64-bit Sum-Product)
   - Classical Normalized Min-Sum (NMS, $\alpha=0.75$)
   - Standalone DAQ-BP (5-bit Integer)
   - Golden Standard BP-OSD (OSD-CS order 10)
   - Hybrid DAQ-BP + OSD Fallback
5. **Hardware Latency & OSD Inversion Reduction Visualization:** Generates publication-grade plots illustrating how DAQ-BP eliminates $>93\%$ of $O(N^3)$ matrix inversions.


In [ ]:
import sys, os, math, time, random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure local package is in sys.path
sys.path.append(os.path.abspath('../src'))
from gross_code import build_gross_code, verify_css
from decoder import DAQBPDecoder, FloatBPDecoder, NMSDecoder
from ldpc import bposd_decoder

# Set plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
print("All dependencies successfully loaded.")


## 1. Code Construction: IBM $[[144, 12, 12]]$ Bivariate Bicycle Code

The code is defined over the group algebra $\mathbb{F}_2[x, y] / (x^{12} - 1, y^6 - 1)$ by two commuting polynomials:
$$A(x, y) = x^3 + y + y^2, \quad B(x, y) = y^3 + x + x^2$$
The check matrices are constructed as $H_X = [M_A \mid M_B]$ and $H_Z = [M_B^T \mid M_A^T]$.


In [ ]:
H_X, H_Z = build_gross_code()
m_X, n_data = H_X.shape

is_css_valid = verify_css(H_X, H_Z)

print(f"Parity Check Matrix H_X Shape: {H_X.shape} (72 check nodes, 144 data qubits)")
print(f"Parity Check Matrix H_Z Shape: {H_Z.shape} (72 check nodes, 144 data qubits)")
print(f"CSS Orthogonality Condition H_X * H_Z^T == 0 (mod 2): {is_css_valid}")
assert is_css_valid, "CSS check failed!"

# Visualize the sparsity pattern of the Gross Code check matrix
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].spy(H_X, markersize=1.8, color='#1f77b4')
axes[0].set_title(r"Structure of $H_X = [M_A \mid M_B]$ (IBM [[144, 12, 12]])")
axes[0].set_xlabel("Data Qubit Index (0..143)")
axes[0].set_ylabel("Check Node Index (0..71)")

axes[1].spy(H_Z, markersize=1.8, color='#d62728')
axes[1].set_title(r"Structure of $H_Z = [M_B^T \mid M_A^T]$")
axes[1].set_xlabel("Data Qubit Index (0..143)")
axes[1].set_ylabel("Check Node Index (0..71)")

plt.tight_layout()
plt.show()


## 2. 5-Bit Quantization Architecture (Zero DSP Multipliers)

Continuous floating-point Belief Propagation computes check node updates using transcendental hyperbolic functions:
$$m_{c \to v} = 2 \operatorname{atanh}\left( (-1)^s \prod_{v' \neq v} \tanh(m_{v' \to c} / 2) \right)$$
In hardware, computing $\tanh$ and multi-operand floating-point products saturates FPGA DSP blocks.

DAQ-BP quantizes all message polarities to signed 5-bit integers in $[-15, +15]$:
- Scale factor $S = 4.0$ (representing values in steps of $0.25$).
- Hyperbolic functions are evaluated via static 32-entry lookup tables (LUTs) requiring $<18$ LUTs per processing unit.
- Multiplications are replaced by integer bit-shifts ($\gg 5$).


In [ ]:
decoder_sample = DAQBPDecoder(H_X, max_iter=35)

q_vals = np.arange(-15, 16)
tanh_lut_vals = decoder_sample.tanh_lut
atanh_lut_vals = decoder_sample.atanh_lut

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

ax1.step(q_vals, tanh_lut_vals, where='mid', color='#2ca02c', linewidth=2, label="5-bit Quantized LUT")
smooth_q = np.linspace(-15, 15, 200)
ax1.plot(smooth_q, np.tanh(smooth_q / 8.0) * 32, 'k--', alpha=0.5, label="Continuous 64-bit tanh")
ax1.set_title("5-bit Quantized tanh LUT (Check Unit)")
ax1.set_xlabel("Quantized Input LLR q in [-15, +15]")
ax1.set_ylabel("Fixed-Point Value (Q5 scale /32)")
ax1.legend()
ax1.grid(True, alpha=0.3)

t_vals = np.arange(-32, 33)
ax2.step(t_vals, atanh_lut_vals, where='mid', color='#9467bd', linewidth=2, label="5-bit Inverse atanh LUT")
ax2.set_title("5-bit Inverse atanh LUT")
ax2.set_xlabel("Product Argument in [-32, +32]")
ax2.set_ylabel("Output Message LLR")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 3. Live Decoding Walkthrough: Resolving a Multi-Qubit Error

We inject a random Pauli-X weight-3 error pattern onto data qubits and compare how DAQ-BP and Float-BP resolve the syndrome.


In [ ]:
np.random.seed(1337)
injected_error = np.zeros(n_data, dtype=int)
# Inject errors on qubits 12, 45, 98
injected_error[[12, 45, 98]] = 1
syndrome = (H_X @ injected_error) % 2

print(f"Injected Error Weight: {np.sum(injected_error)} (Qubits: {np.where(injected_error)[0]})")
print(f"Syndrome Weight: {np.sum(syndrome)} defects out of 72 checks")

daq_dec = DAQBPDecoder(H_X, max_iter=35)
float_dec = FloatBPDecoder(H_X, max_iter=35)

t0 = time.perf_counter()
daq_est, daq_it, daq_conv = daq_dec.decode(list(syndrome), p=0.01)
daq_time_us = (time.perf_counter() - t0) * 1e6

t0 = time.perf_counter()
float_est, float_it, float_conv = float_dec.decode(list(syndrome), p=0.01)
float_time_us = (time.perf_counter() - t0) * 1e6

# Check syndrome satisfaction
daq_satisfied = np.array_equal((H_X @ np.array(daq_est)) % 2, syndrome)
float_satisfied = np.array_equal((H_X @ np.array(float_est)) % 2, syndrome)

print("-" * 65)
print(f"DAQ-BP (5-bit): Converged={daq_conv} in {daq_it} iters | Residual Syndrome=0: {daq_satisfied} | Python Time: {daq_time_us:.1f} us")
print(f"Float-BP:       Converged={float_conv} in {float_it} iters | Residual Syndrome=0: {float_satisfied} | Python Time: {float_time_us:.1f} us")
print("-" * 65)


## 4. Live Multi-Decoder Benchmark Across Error Rates

We evaluate:
1. **Float-BP:** Standard 64-bit sum-product BP.
2. **NMS:** Normalized Min-Sum ($\alpha=0.75$).
3. **DAQ-BP:** Our 5-bit integer quantized BP.
4. **BP-OSD:** Golden standard floating-point BP with Order-10 OSD Combination Sweep.
5. **DAQ-BP+OSD:** Fast 5-bit integer hardware front-end with OSD fallback on timeouts.


In [ ]:
noise_rates = [0.005, 0.010, 0.020, 0.030, 0.040]
shots_per_rate = 300   # Sufficient for live notebook execution

results = {
    'Float-BP': [], 'NMS': [], 'DAQ-BP': [], 'BP-OSD': [], 'DAQ-BP+OSD': []
}
daq_conv_rates = []
osd_fallback_rates = []
daq_converged_iters = []

LOGICAL_X = list(np.where(H_Z[0])[0])

print(f"Starting Monte Carlo Evaluation ({shots_per_rate} shots/point across {len(noise_rates)} rates)...")
t_start = time.time()

for p in noise_rates:
    t_p = time.time()
    random.seed(int(p * 100000))
    errors = [[1 if random.random() < p else 0 for _ in range(n_data)] for _ in range(shots_per_rate)]
    
    decoders = {
        'Float-BP': FloatBPDecoder(H_X, max_iter=35),
        'NMS': NMSDecoder(H_X, alpha=0.75, max_iter=35),
        'DAQ-BP': DAQBPDecoder(H_X, max_iter=35),
    }
    bposd = bposd_decoder(
        H_X, error_rate=max(p, 1e-4), max_iter=35, bp_method='ms',
        ms_scaling_factor=0.75, osd_method='osd_cs', osd_order=10
    )
    
    log_errs = {k: 0 for k in results}
    daq_conv_cnt = 0
    daq_it_list = []
    osd_fallbacks = 0
    
    for err in errors:
        syn = (H_X @ np.array(err)) % 2
        syn_list = list(syn)
        
        # Run decoders
        f_est, _, _ = decoders['Float-BP'].decode(syn_list, p)
        n_est, _, _ = decoders['NMS'].decode(syn_list, p)
        d_est, d_it, d_conv = decoders['DAQ-BP'].decode(syn_list, p)
        osd_est = bposd.decode(syn)
        
        if d_conv:
            daq_conv_cnt += 1
            daq_it_list.append(d_it)
            hybrid_est = d_est
        else:
            osd_fallbacks += 1
            hybrid_est = osd_est
            
        def check_fail(est):
            res = [(err[i] ^ int(est[i])) for i in range(n_data)]
            return (sum(res[q] for q in LOGICAL_X) % 2) != 0

        if check_fail(f_est): log_errs['Float-BP'] += 1
        if check_fail(n_est): log_errs['NMS'] += 1
        if check_fail(d_est): log_errs['DAQ-BP'] += 1
        if check_fail(osd_est): log_errs['BP-OSD'] += 1
        if check_fail(hybrid_est): log_errs['DAQ-BP+OSD'] += 1

    for k in results:
        results[k].append((log_errs[k] / shots_per_rate) * 100.0)
        
    daq_conv_rates.append((daq_conv_cnt / shots_per_rate) * 100.0)
    osd_fallback_rates.append((osd_fallbacks / shots_per_rate) * 100.0)
    daq_converged_iters.append(np.mean(daq_it_list) if daq_it_list else 1.0)
    
    print(f"  p = {p:.3f} completed in {time.time()-t_p:.1f}s | BP-OSD LER: {results['BP-OSD'][-1]:.2f}% | DAQ-BP+OSD LER: {results['DAQ-BP+OSD'][-1]:.2f}% | OSD Avoided: {100-osd_fallback_rates[-1]:.1f}%")

print(f"Total benchmark finished in {time.time()-t_start:.1f}s.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Plot 1: Logical Error Rate Comparison
ax1 = axes[0]
ax1.plot(noise_rates, results['Float-BP'], 'o--', color='#1f77b4', label='Float-BP (64-bit MS)', linewidth=1.8)
ax1.plot(noise_rates, results['NMS'], 's--', color='#ff7f0e', label='NMS (alpha=0.75)', linewidth=1.8)
ax1.plot(noise_rates, results['DAQ-BP'], '^--', color='#2ca02c', label='DAQ-BP (5-bit Integer Standalone)', linewidth=2.0)
ax1.plot(noise_rates, results['BP-OSD'], 'd-', color='#d62728', label='BP-OSD Reference (Float)', linewidth=2.2)
ax1.plot(noise_rates, results['DAQ-BP+OSD'], '*-', color='#9467bd', label='DAQ-BP + OSD Fallback (Hybrid)', linewidth=2.5)

ax1.set_yscale('log')
ax1.set_xlabel("Physical Error Rate p (Code-Capacity Pauli-X)")
ax1.set_ylabel("Logical Error Rate (%) [Log Scale]")
ax1.set_title("Logical Error Rate Suppression on IBM [[144, 12, 12]]")
ax1.legend(frameon=True)
ax1.grid(True, which="both", ls="--", alpha=0.4)

# Plot 2: OSD Avoidance & Hardware Acceleration Efficiency
ax2 = axes[1]
inversions_avoided = [100.0 - f for f in osd_fallback_rates]

bars = ax2.bar([f"p={p}" for p in noise_rates], inversions_avoided, color='#2ca02c', alpha=0.85, width=0.55, edgecolor='black', linewidth=1)
ax2.set_xlabel("Physical Error Rate p")
ax2.set_ylabel("Matrix Inversions Avoided by DAQ-BP (%)")
ax2.set_title("Elimination of O(N^3) Matrix Inversions via 5-Bit Front-End")
ax2.set_ylim(0, 105)
ax2.grid(True, axis='y', ls="--", alpha=0.4)

# Add text labels on bars
for bar in bars:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width() / 2., height + 1.5, f'{height:.1f}%',
             ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../paper/benchmark_verification.png', dpi=300)
plt.show()


## 5. Hardware Microarchitecture Latency Summary

Based on the synthesizable SystemVerilog models in `rtl/daq_bp_node.sv`:
- **Datapath Pipeline:** 4 pipeline stages running at $400\,\text{MHz}$ ($2.5\,\text{ns}$ clock $\to 10\,\text{ns}$ per iteration).
- **Mean Converged Latency:** At operational rates ($p \le 0.01$), average iteration count is $\approx 1.05$, yielding **$10.5\,\text{ns}$** syndrome resolution time.
- **Worst-Case Tail Latency:** Strictly bounded at $35 \times 10\,\text{ns} = \mathbf{350\,\text{ns}}$, fitting well within the microsecond coherence window of superconducting qubits.
